# 05. Which part of the gap costs accuracy? (Step 2 diagnostics)

Three cheap tests, each tied to a finding of notebook 02. All use source-only CTR-GCN, seed 0, and the same 4 subject-disjoint folds as the benchmark.

1. **Label shift** (inference only): re-weight the source model's posteriors by the target class prior estimated with EM on the *unlabeled* training subjects of each fold (Saerens et al., 2002). The `true_prior` row uses labels and is only an upper bound.
2. **Fixed-window shortcut** (one ~10 min training): random sub-windows of 30-100% of each InHARD-DT clip (`source_aug='temporal'`).
3. **Bimanual vs one-handed** (inference + ~10 min training each): remove left-arm motion relative to the shoulder, at test time only, and at both train and test time (`freeze_left=True`).

Total: about 35 minutes. Before running, close or restart the other notebook kernels so the GPU is free.

**What to send back:** the tables of cells 3 and 5 (or just say it is done and I will read the CSV).

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

from picda import config as C
from picda.data import freeze_left_arm
from picda.diagnostics import fold_table, load_model, real_probs, summarize, em_prior
from picda.experiments import Benchmark
from picda.train import TrainConfig

DIAG = C.OUTPUT_DIR / 'diagnostics'
BASE_CKPT = C.OUTPUT_DIR / 'benchmark_ctrgcn' / 'checkpoints' / 'source_only_seed0.pt'
assert BASE_CKPT.exists(), BASE_CKPT
tables, probs_store = [], {}

## 1. Inference-only tests on the existing source-only model

In [2]:
bench = Benchmark(TrainConfig(seeds=(0,)), DIAG)
base = load_model(bench, BASE_CKPT)

P, y = real_probs(bench, base)
probs_store['source_only'] = P
tables.append(fold_table(bench, P, y, 'source_only'))

P_fl, _ = real_probs(bench, base, transform=freeze_left_arm)
probs_store['source_only + test-time left freeze'] = P_fl
tables.append(fold_table(bench, P_fl, y, 'source_only + test-time left freeze'))

print(summarize(pd.concat(tables)).to_string())

# How well does EM recover the real class mix? (fold 0)
src_prior = np.bincount(bench.y_dt, minlength=C.NUM_CLASSES) / len(bench.y_dt)
tr, _ = bench._real_split(0)
print()
print(pd.DataFrame({'DT prior': src_prior, 'EM estimate': em_prior(P[tr], src_prior),
                    'true real prior': np.bincount(y[tr], minlength=C.NUM_CLASSES) / len(tr)},
                   index=C.CLASS_NAMES).mul(100).round(1).to_string())

Loaded 4696 DT clips (12 subjects) and 4803 InHARD clips (16 subjects) in 7s
                                                                acc  macro_f1  macro_f1_std  bal_acc  prior_L1_err
model                               correction                                                                    
source_only                         plain                     50.80     41.37          3.55    44.70           NaN
                                    em_prior                  48.71     39.01          3.30    39.09          0.47
                                    true_prior (upper bound)  53.09     41.47          3.73    41.47           NaN
source_only + test-time left freeze plain                     39.02     31.01          2.13    30.78           NaN
                                    em_prior                  34.56     26.14          2.40    25.30          0.76
                                    true_prior (upper bound)  41.81     30.75          2.42    29.14           NaN

  

## 2. Retrained source-only variants (about 10 minutes each)

Resumable: re-running skips variants whose checkpoint already exists.

In [3]:
VARIANTS = [
    ('temporal', dict(source_aug='temporal')),
    ('noleft', dict(freeze_left=True)),
    ('temporal + noleft', dict(source_aug='temporal', freeze_left=True)),
]
for name, kw in VARIANTS:
    cfg = TrainConfig(seeds=(0,), **kw)
    b = Benchmark(cfg, C.OUTPUT_DIR / ('diag_' + name.replace(' + ', '_')))
    b.run(['source_only'], seeds=[0])
    m = load_model(b, b._ckpt('source_only_seed0'))
    Pv, yv = real_probs(b, m, transform=freeze_left_arm if cfg.freeze_left else None)
    assert (yv == y).all()
    probs_store[name] = Pv
    tables.append(fold_table(b, Pv, yv, name))

allt = pd.concat(tables)
allt.to_csv(DIAG / 'diagnostics_per_fold.csv', index=False)
summary = summarize(allt)
summary.to_csv(DIAG / 'diagnostics_summary.csv')
print(summary.to_string())

Loaded 4696 DT clips (12 subjects) and 4803 InHARD clips (16 subjects) in 4s

================ seed 0 ================
  [source_only s0] step 200/3650 | loss=2.3704 | train_acc=0.3480 | 38s
  [source_only s0] step 400/3650 | loss=1.2296 | train_acc=0.5852 | 68s
  [source_only s0] step 600/3650 | loss=0.9089 | train_acc=0.6937 | 98s
  [source_only s0] step 800/3650 | loss=0.7184 | train_acc=0.7559 | 128s
  [source_only s0] step 1000/3650 | loss=0.6118 | train_acc=0.7915 | 159s
  [source_only s0] step 1200/3650 | loss=0.5589 | train_acc=0.8071 | 189s
  [source_only s0] step 1400/3650 | loss=0.5055 | train_acc=0.8281 | 219s
  [source_only s0] step 1600/3650 | loss=0.4567 | train_acc=0.8427 | 250s
  [source_only s0] step 1800/3650 | loss=0.4000 | train_acc=0.8624 | 280s
  [source_only s0] step 2000/3650 | loss=0.3628 | train_acc=0.8743 | 311s
  [source_only s0] step 2200/3650 | loss=0.3316 | train_acc=0.8854 | 341s
  [source_only s0] step 2400/3650 | loss=0.2980 | train_acc=0.9005 | 371s


## 3. Per-class view (no prior correction, all 16 subjects)

Fixed-window share of each class in InHARD-DT is shown for reference: if the shortcut hypothesis holds, `temporal` should help most where that share is high.

In [4]:
fixed_share = {'Take subsystem': 99, 'Take component': 97, 'Put down component': 97, 'Turn sheets': 86,
               'Take screwdriver': 75, 'Put down screwdriver': 70, 'Take measuring rod': 67,
               'Picking in front': 57, 'Picking left': 47, 'Consult sheets': 2, 'Assemble system': 2,
               'Put down measuring rod': 1, 'Put down subsystem': 1}
pc = pd.DataFrame({k: f1_score(y, v.argmax(1), labels=range(C.NUM_CLASSES), average=None, zero_division=0) * 100
                   for k, v in probs_store.items()}, index=C.CLASS_NAMES)
pc.insert(0, 'DT fixed-window %', pd.Series(fixed_share))
pc = pc.sort_values('DT fixed-window %', ascending=False)
pc.to_csv(DIAG / 'diagnostics_per_class.csv')
print(pc.round(1).to_string())

                        DT fixed-window %  source_only  source_only + test-time left freeze  temporal  noleft  temporal + noleft
Take subsystem                         99         16.1                                 15.4      25.8     8.5               12.9
Take component                         97         15.2                                 13.2       5.9    14.6                2.7
Put down component                     97         67.7                                 65.8      40.0    57.3               39.5
Turn sheets                            86         37.1                                 20.8      40.9    22.5               32.2
Take screwdriver                       75         64.5                                 62.1      52.5    59.2               53.1
Put down screwdriver                   70         38.4                                 35.9      42.9    41.3               49.8
Take measuring rod                     67         24.9                                  4.1      